# Project 2: Transformers

This project is part of the NLP module held in the spring of 2026. Three transformer models are compared, answering physical common sense tasks with the PIQA dataset. 

- **Randomly Initialized Transformer** 
- **Pretrained Transformer** not pretrained or finetuned on PIQA dataset
- **LLM (1B+ Parameters)** same hyperparameters

**Dataset**  
"PIQA: Reasoning about Physical Commonsense in Natural Language" — a binary choice task
where a model selects the more physically plausible solution to a given goal.  
Source: [https://arxiv.org/abs/1911.11641](https://arxiv.org/abs/1911.11641)

**Tools** 
- Course Materials
- Documentations (mostly of imported dependencies)
    - apxml
    - NLTK
    - PyTorch
    - skikit-learn
- Claude AI for the following tasks:
    - Helping formulate and clarify reasoning
    - General coding assistance
- Regex101
- Huggingface

**Weights & Biases**  
All experimental runs are logged and published in the
# TODO report

**Notebook structure**
1. Introduction
2. Setup
3. Preprocessing
4. Model
5. Training
6. Evaluation
7. Interpretation


## Setup

In [1]:
!pip install \
    datasets==4.8.4 \
    numpy==2.4.4 \
    datetime==6.0.0 \
    transformers==5.6.2 \
    torch==2.11.0 \
    wandb==0.25.1


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from datasets import load_dataset
from datetime import datetime
from transformers import AutoTokenizer, BertConfig, BertModel
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import numpy as np
import wandb
import re
import torch

In [3]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
generator = torch.Generator()
generator.manual_seed(SEED)

In [4]:
TS = datetime.now().strftime("%Y%m%d_%H%M%S")

In [5]:
wandb_project = "nlp-project2-piqa"

## Preprocessing

In [6]:
train_split = load_dataset("ybisk/piqa", split="train[:-1000]", revision='refs/convert/parquet')
valid_split = load_dataset("ybisk/piqa", split="train[-1000:]", revision='refs/convert/parquet')
test_split = load_dataset("ybisk/piqa", split="validation", revision='refs/convert/parquet')

### Feature Selection

In [7]:
COL_GOAL = 'goal'
COL_SOL1 = 'sol1'
COL_SOL2 = 'sol2'
COL_LABEL = 'label'

### Filter HTML Elements

In [8]:
# regex source: https://apxml.com/courses/nlp-fundamentals/chapter-1-nlp-text-processing-techniques/handling-text-noise
# verified with: https://regex101.com
regex_pattern = re.compile(r'<[^>]+>', re.IGNORECASE)
html_elements = 0

for split in [train_split, valid_split, test_split]:
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_GOAL])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL1])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL2])))

print(f"Number of HTML elements found: {html_elements}")

Number of HTML elements found: 0


### Input Format

In [9]:
MAX_INPUT_LENGTH = 553 # value found by analysis further down

COL_INPUT1 = 'input1'
COL_INPUT2 = 'input2'

ATTENTION_MASK = 'attention_mask'
INPUT_IDS = 'input_ids'
TOKEN_TYPE_IDS = 'token_type_ids'

COL_INPUT1_ATTENTION_MASK = f"{COL_INPUT1}_{ATTENTION_MASK}"
COL_INPUT1_INPUT_IDS = f"{COL_INPUT1}_{INPUT_IDS}"
COL_INPUT1_TOKEN_TYPE_IDS = f"{COL_INPUT1}_{TOKEN_TYPE_IDS}"
COL_INPUT2_ATTENTION_MASK = f"{COL_INPUT2}_{ATTENTION_MASK}"
COL_INPUT2_INPUT_IDS = f"{COL_INPUT2}_{INPUT_IDS}"
COL_INPUT2_TOKEN_TYPE_IDS  = f"{COL_INPUT2}_{TOKEN_TYPE_IDS}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_row(row):
    tokenized1 = tokenizer(row[COL_GOAL], row[COL_SOL1], truncation=True, max_length=MAX_INPUT_LENGTH)
    tokenized2 = tokenizer(row[COL_GOAL], row[COL_SOL2], truncation=True, max_length=MAX_INPUT_LENGTH)
    return {
        COL_LABEL: row[COL_LABEL],
        COL_INPUT1_ATTENTION_MASK: tokenized1[ATTENTION_MASK],
        COL_INPUT1_INPUT_IDS: tokenized1[INPUT_IDS],
        COL_INPUT1_TOKEN_TYPE_IDS: tokenized1[TOKEN_TYPE_IDS],
        COL_INPUT2_ATTENTION_MASK: tokenized2[ATTENTION_MASK],
        COL_INPUT2_INPUT_IDS: tokenized2[INPUT_IDS],
        COL_INPUT2_TOKEN_TYPE_IDS: tokenized2[TOKEN_TYPE_IDS],
    }

train_processed = train_split.map(preprocess_row, remove_columns=train_split.column_names, batched=False)
valid_processed = valid_split.map(preprocess_row, remove_columns=valid_split.column_names, batched=False)
test_processed = test_split.map(preprocess_row,  remove_columns=test_split.column_names, batched=False)

print(train_processed[0])

{'label': 1, 'input1_attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'input1_input_ids': [101, 2043, 16018, 12136, 1010, 2043, 2009, 1005, 1055, 3201, 1010, 2017, 2064, 102, 10364, 2009, 3031, 1037, 5127, 102], 'input1_token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1], 'input2_attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'input2_input_ids': [101, 2043, 16018, 12136, 1010, 2043, 2009, 1005, 1055, 3201, 1010, 2017, 2064, 102, 10364, 2009, 2046, 1037, 15723, 102], 'input2_token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]}


### Length Analysis

In [10]:
all_input_lengths = (
    [len(row[COL_INPUT1_ATTENTION_MASK]) for row in train_processed] +
    [len(row[COL_INPUT2_ATTENTION_MASK]) for row in train_processed] +
    [len(row[COL_INPUT1_ATTENTION_MASK]) for row in valid_processed] +
    [len(row[COL_INPUT2_ATTENTION_MASK]) for row in valid_processed] +
    [len(row[COL_INPUT1_ATTENTION_MASK]) for row in test_processed] +
    [len(row[COL_INPUT2_ATTENTION_MASK]) for row in test_processed]
)

print(f"Max length of an input field: {np.max(all_input_lengths)}") 

Max length of an input field: 553


### Data Loader

In [11]:
BATCH_SIZE = 128

class PiqaDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        return {
            COL_INPUT1_INPUT_IDS: torch.tensor(row[COL_INPUT1_INPUT_IDS]),
            COL_INPUT1_ATTENTION_MASK: torch.tensor(row[COL_INPUT1_ATTENTION_MASK]),
            COL_INPUT1_TOKEN_TYPE_IDS: torch.tensor(row[COL_INPUT1_TOKEN_TYPE_IDS]),
            COL_INPUT2_INPUT_IDS: torch.tensor(row[COL_INPUT2_INPUT_IDS]),
            COL_INPUT2_ATTENTION_MASK: torch.tensor(row[COL_INPUT2_ATTENTION_MASK]),
            COL_INPUT2_TOKEN_TYPE_IDS: torch.tensor(row[COL_INPUT2_TOKEN_TYPE_IDS]),
            COL_LABEL: torch.tensor(row[COL_LABEL]),
        }
    
def pad_sequence(sequences):
    max_length_in_batch = max(len(s) for s in sequences)
    return torch.stack([
        torch.nn.functional.pad(s, (0, max_length_in_batch - len(s))) for s in sequences
    ])

def collate_fn(batch):
    input1_ids = pad_sequence([item[COL_INPUT1_INPUT_IDS] for item in batch])
    input1_attention_mask = pad_sequence([item[COL_INPUT1_ATTENTION_MASK] for item in batch])
    input1_token_type_ids = pad_sequence([item[COL_INPUT1_TOKEN_TYPE_IDS] for item in batch])
    input2_ids = pad_sequence([item[COL_INPUT2_INPUT_IDS] for item in batch])
    input2_attention_mask = pad_sequence([item[COL_INPUT2_ATTENTION_MASK] for item in batch])
    input2_token_type_ids = pad_sequence([item[COL_INPUT2_TOKEN_TYPE_IDS] for item in batch])
    labels = torch.tensor([item[COL_LABEL] for item in batch], dtype=torch.long)

    return (
        input1_ids, 
        input1_attention_mask, 
        input1_token_type_ids,
        input2_ids, 
        input2_attention_mask, 
        input2_token_type_ids,
        labels
    )

# The training set has to be shuffled to ensure random order in training which makes training more stable.  
train_loader = DataLoader(PiqaDataset(train_processed), batch_size=BATCH_SIZE, collate_fn=collate_fn, shuffle=True, generator=generator)
# Test and Validation should not be shuffled to ensure reproducibility and consistency of the model.
valid_loader = DataLoader(PiqaDataset(valid_processed), batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(PiqaDataset(test_processed), batch_size=BATCH_SIZE, collate_fn=collate_fn)

## Model

### Bert Classifier
to ensure same base architecture

In [12]:
class BertTransformerClassifier(nn.Module):

    def __init__(self, bert: BertModel, dropout_p: float):
        super().__init__()
        self.bert = bert
        hidden_size = bert.config.hidden_size
        input_hidden_dim = 2 * hidden_size
        classifier_hidden_dim = input_hidden_dim // 2

        self.classifier = nn.Sequential(
            nn.Linear(input_hidden_dim, classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_p),
            nn.Linear(classifier_hidden_dim, 2),
        )

    def encode(self, input_ids, attention_mask, token_type_ids):
        output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # output: (B, L, 768)
        # B: for the whole batch size
        # L: length of input (nr of tokens)
        # 768 dim vector which is a summary of the whole input but biased by the current token -> [CLS] does not influence the summary -> all tokens are weighed based on the importance of the information they carry -> we can use this dim only
        cls_vector = output.last_hidden_state[:, 0, :]
        return cls_vector
    
    def forward(self, 
                input1_ids, input1_attention_mask, input1_token_type_ids, 
                input2_ids, input2_attention_mask, input2_token_type_ids):
        cls1 = self.encode(input1_ids, input1_attention_mask, input1_token_type_ids)
        cls2 = self.encode(input2_ids, input2_attention_mask, input2_token_type_ids)
        # cls dim: 768
        # combined dim: 2 * 768
        combined = torch.cat([cls1, cls2], dim=-1)
        logits = self.classifier(combined)
        return logits

### Randomly Initialized Transformer

In [13]:
# use default config to match pretrained transformer
# defaults described here: https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertConfig
bert_config_random = BertConfig()

bert_random = BertModel(bert_config_random)

### Pretrained Transformer

In [14]:
bert_pretrained = BertModel.from_pretrained("bert-base-uncased") 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### LLM (1B+ Parameters)

## Training

In [15]:
SKIP_TRAINING = True
MODEL1_NAME = f"random_{TS}"
MODEL2_NAME = f"pretrained_{TS}"
SWEEP_COUNT = 5

In [16]:
training_config = {
    'max_epochs': 30,
    'patience': 5,
}

# use same model architecture in model 1 and 2
# todo config
def create_sweep_config(model_name):
    return {
        "method": "bayes",
        "metric": {"name": f"{model_name}/valid_acc", "goal": "maximize"},
        "parameters": {
            "lr":                  {"values": [1e-3, 1e-4, 1e-5]},
            "weight_decay":        {"values": [1e-3, 1e-4, 1e-5]},
            "dropout_probability": {"values": [0.1, 0.3, 0.5]},
        },
    }

In [17]:
if not SKIP_TRAINING:
    wandb.login()
else: 
    print("training skipped")

training skipped


In [18]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, total_correct, total = 0, 0, 0

    for input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt, labels in loader:
        optimizer.zero_grad()
        logits = model(input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0, 0, 0

    with torch.no_grad():
        for input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt, labels in loader:
            logits = model(input1_ids, input1_mask, input1_tt, input2_ids, input2_mask, input2_tt)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(dim=-1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, total_correct / total

def train_model(model, config, model_name, wandb_run, train_loader, valid_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config['lr'],
        weight_decay=config['weight_decay']
    )
    best_valid_acc = 0
    patience_counter = 0
    
    for epoch in range(config['max_epochs']):
        print("____________________________________")
        print(f"Epoch {epoch + 1}/{config['max_epochs']}")
        
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        valid_loss, valid_acc = eval_epoch(model, valid_loader, criterion)
        
        print(f"Train loss: {train_loss:.4f} - Train acc: {train_acc:.4f} | Valid loss: {valid_loss:.4f} - Valid acc: {valid_acc:.4f}")
        
        wandb_run.log({
            f"{model_name}/epoch": epoch + 1, 
            f"{model_name}/train_loss": train_loss,
            f"{model_name}/train_acc": train_acc,
            f"{model_name}/valid_loss": valid_loss,
            f"{model_name}/valid_acc": valid_acc,
        })
        
        if valid_acc > best_valid_acc:
            best_valid_acc = valid_acc
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': config
            }, config['model_path'])
            print(f"New best model saved (valid_acc: {valid_acc:.4f})")
        else:
            patience_counter += 1
            print(f"No improvement ({patience_counter}/{config['patience']})")

        if patience_counter >= config['patience']:
            print("Early stopping triggered")
            break

In [19]:
def sweep_run_model1():
    with wandb.init(project=wandb_project, config=training_config, group=f"random_{TS}") as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{MODEL1_NAME}_lr{wandb.config.lr}_wd{wandb.config.weight_decay}_dp{wandb.config.dropout_probability}_{wandb_run.id[:4]}"
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        model = BertTransformerClassifier(bert=bert_random, dropout_p=wandb_config.dropout_probability)

        train_model(model, config, MODEL1_NAME, wandb_run, train_loader, valid_loader)
        
        
if not SKIP_TRAINING:
    sweep_model1 = wandb.sweep(sweep=create_sweep_config(MODEL1_NAME), project=wandb_project)
    
    wandb.agent(sweep_model1, function=sweep_run_model1, count=SWEEP_COUNT)
else: 
    print("training skipped")

training skipped


In [20]:
def sweep_run_model2():
    with wandb.init(project=wandb_project, config=training_config, group=f"pretrained_{TS}") as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{MODEL2_NAME}_lr{wandb.config.lr}_wd{wandb.config.weight_decay}_dp{wandb.config.dropout_probability}_{wandb_run.id[:4]}"
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        model = BertTransformerClassifier(bert=bert_pretrained, dropout_p=wandb_config.dropout_probability)
        
        train_model(model, config, MODEL2_NAME, wandb_run, train_loader, valid_loader)
        

if not SKIP_TRAINING:
    sweep_model2 = wandb.sweep(sweep=create_sweep_config(MODEL2_NAME), project=wandb_project)
    
    wandb.agent(sweep_model2, function=sweep_run_model2, count=SWEEP_COUNT)
else: 
    print("training skipped")

training skipped


## Evaluation

## Interpretation